# MS-Dialog Phase 1 — Single-Turn Experiment

**Research question**: When a tech-support specialist model is uncertain about a user's problem, does it ask an *Epistemic* clarifying question (knowledge gap about this specific case) or an *Aleatoric* one (inherent ambiguity in the user's description)?  How does this behaviour vary across models and domains?

**Pipeline per record:**
1. Model sees `category` + `title` + `original_question` → preliminary solution + CQ + confidence
2. User simulator answers the CQ using the synthesised situation summary (hidden from model)
3. Model gives updated solution + updated confidence

**Dataset:** `msdialog_100.jsonl` — 100 MS-Dialog tech-support cases, 19 categories

**CQ classification** is done separately via `run_msdialog_judge.py`

In [ ]:
import sys
from pathlib import Path

ROOT       = Path("../../").resolve()
sys.path.insert(0, str(ROOT))

from config import SIMULATOR_MODEL_ID, MSDIALOG_GEMINI_MODEL_ID

DATASET      = "ms-dialog"
DATASETS_DIR = ROOT / "datasets" / DATASET
PROMPTS_DIR  = ROOT / "prompts"  / DATASET
MODEL_ID     = MSDIALOG_GEMINI_MODEL_ID          # specialist / clinician model
OUTPUTS_DIR  = ROOT / "outputs" / DATASET / MODEL_ID

CASES_PATH       = DATASETS_DIR / "msdialog_100.jsonl"
INSTRUCTION_FILE = PROMPTS_DIR  / "phase1_instruction.txt"
OUTPUT_CSV       = OUTPUTS_DIR  / "phase1_singleturn_results.csv"

REQUEST_INTERVAL = 1.0

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Specialist : {MODEL_ID}")
print(f"Simulator  : {SIMULATOR_MODEL_ID}")
print(f"Cases      : {CASES_PATH}")
print(f"Output CSV : {OUTPUT_CSV}")

In [ ]:
import importlib
import json
import logging

import src.utils as utils_mod
import src.providers as providers_mod
import src.pipeline as pipeline_mod
importlib.reload(utils_mod)
importlib.reload(providers_mod)
importlib.reload(pipeline_mod)

from src.utils import load_dotenv
from src.providers import GeminiProvider
from src.pipeline import MsDialogPhase1Pipeline, UserSimulator, MSDIALOG_TURN_0_SCHEMA

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded.")

## Smoke Test

In [ ]:
provider = GeminiProvider(model_id=MODEL_ID)
resp = provider.call(
    system_instruction="You are a helpful assistant.",
    user_message="Reply with exactly: SMOKE TEST PASSED",
    temperature=0.0, max_tokens=5000,
)
assert resp and "SMOKE" in resp.upper(), f"Smoke test failed: {resp!r}"
print(f"Specialist smoke test PASSED: {resp.strip()}")

sim_provider = GeminiProvider(model_id=SIMULATOR_MODEL_ID)
resp2 = sim_provider.call(
    system_instruction="You are a helpful assistant.",
    user_message="Reply with exactly: SMOKE TEST PASSED",
    temperature=0.0, max_tokens=5000,
)
assert resp2 and "SMOKE" in resp2.upper(), f"Simulator smoke test failed: {resp2!r}"
print(f"Simulator  smoke test PASSED: {resp2.strip()}")

## Load Dataset

In [ ]:
with open(CASES_PATH, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(records)} records")
print(f"Fields: {list(records[0].keys())}")

from collections import Counter
cats = Counter(r["category"] for r in records)
print(f"\nCategories ({len(cats)} unique):")
for cat, n in cats.most_common():
    print(f"  {cat:35s}: {n}")

## Dry Run — Single Record
Verify the two-turn flow on one record before running everything.

In [ ]:
from src.pipeline import _format_problem, _MSDIALOG_FINAL_INSTRUCTION, MSDIALOG_FINAL_SCHEMA
from src.utils import parse_json_response
import json as _json

simulator = UserSimulator(sim_provider)
instruction = INSTRUCTION_FILE.read_text(encoding="utf-8").strip()

test = records[0]
print(f"Testing on: {test['id']} | {test['category']}")
print(f"Title: {test['title']}")
print(f"OQ: {test['original_question'][:200]}")
print()

# Turn 0
raw_0 = provider.call(
    system_instruction=instruction,
    user_message=_format_problem(test["title"], test["category"], test["original_question"]),
    temperature=0.0, max_tokens=5000, expect_json=MSDIALOG_TURN_0_SCHEMA,
)
p0 = parse_json_response(raw_0)
print(f"=== TURN 0 ===\n{_json.dumps(p0, indent=2)}")

# Simulate user response
sim_resp = simulator.answer(p0["clarifying_question"], test["simulator_context"])
print(f"\n=== USER RESPONSE ===\n{sim_resp}")

# Turn 1 — final solution
user_msg_1 = (
    f"{_format_problem(test['title'], test['category'], test['original_question'])}\n\n"
    f"Your clarifying question:\n{p0['clarifying_question']}\n\n"
    f"User's answer:\n{sim_resp}\n\n"
    f"Based on this additional information, provide your updated solution."
)
raw_1 = provider.call(
    system_instruction=_MSDIALOG_FINAL_INSTRUCTION,
    user_message=user_msg_1,
    temperature=0.0, max_tokens=5000, expect_json=MSDIALOG_FINAL_SCHEMA,
)
p1 = parse_json_response(raw_1)
print(f"\n=== TURN 1 (FINAL) ===\n{_json.dumps(p1, indent=2)}")
print(f"\n--- ACCEPTED ANSWER ---\n{test['accepted_answer'][:300]}")

## Run Full Experiment
100 cases × 1 CQ round. Resumes automatically if interrupted.

In [ ]:
pipeline = MsDialogPhase1Pipeline(
    provider=provider,
    instruction_file=INSTRUCTION_FILE,
    output_csv=OUTPUT_CSV,
    request_interval=REQUEST_INTERVAL,
    simulator_provider=sim_provider,
)

pipeline.run(records)

## Results Summary

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(OUTPUT_CSV)
valid = df[~df["was_blocked"]]

print(f"Total: {len(df)} | Blocked: {df['was_blocked'].sum()} | Valid: {len(valid)}")
print(f"\nMean preliminary confidence : {valid['preliminary_confidence'].mean():.1f}")
print(f"Mean updated confidence     : {valid['updated_confidence'].mean():.1f}")
print(f"Mean confidence delta       : {(valid['updated_confidence'] - valid['preliminary_confidence']).mean():.1f}")

print("\nSample CQs generated:")
for _, row in valid.head(5).iterrows():
    print(f"  [{row['id']}] {row['clarifying_question'][:100]}")

print("\nNote: CQ classification (EPISTEMIC/ALEATORIC) → run run_msdialog_judge.py")